# TBD Phase 2 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas, DuckDB, and PySpark**.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores and with Spark executors on a cluster.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.

This notebook is an assignment template. It gives you a common structure and helper code, but you must design your own dataset variant, queries, benchmark implementation, and analysis.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your group number,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.

In [1]:
# TODO: Fill this in before submitting.
GROUP_ID = 3
NOTEBOOK_URL = "https://github.com/m-baj/tbd-workshop-1/blob/phase2/notebooks/tbd_phase_2_26L.ipynb"
GROUP_MEMBERS = [
    "Maksymilian Baj, 325144",
    "Adam Filewicz",
    "Jan Lewandowski 325184",
]

assert GROUP_ID is not None, "Set GROUP_ID before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | yes | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | yes, for selected IO/UDF paths | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0 in this lab. Two pandas 3.0 behaviours matter for the benchmark: string columns are no longer inferred as generic `object` dtype by default, and Copy-on-Write is the only mutation model. In addition, compare two Pandas Parquet-reading variants where possible:

- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Prerequisites

Install the required libraries in your notebook environment. If the course image already contains them, this command should be quick. Pandas 3.0 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.


In [5]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb pyspark faker deltalake memory_profiler pyarrow psutil matplotlib seaborn

/home/yannosh/tbd/tbd-workshop-1/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import os
import time
import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from faker import Faker
from memory_profiler import memory_usage
from pyspark.sql import SparkSession

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.12.12
Polars: 1.40.1
Pandas: 3.0.2
DuckDB: 1.5.2
CPU logical cores: 16
RAM GiB: 15.31


## Part 1: Data generation with group variants

Each group works with one assigned synthetic data profile. Use your group number to select the variant card below.

Your dataset does not need to match other groups exactly, but it must satisfy the common schema and benchmarking requirements described in this notebook.

Every group must document:
- dataset profile,
- main benchmark row count, plus any additional stress-test row counts if used,
- physical layout and file format choices,
- library versions,
- query intent,
- benchmark results,
- conclusions.

You may use the helper functions below, but you must adapt the dataset to your assigned variant.


## Variant cards for 16 groups

Choose or assign one variant per group.

| Group | Data profile | Required data feature | Suggested query stress |
|---:|---|---|---|
| 1 | Social media posts | tags or hashtags | explode/list handling, top-k |
| 2 | E-commerce orders | products and order values | join, category aggregation |
| 3 | IoT telemetry | device time series | time filters, rolling/window logic |
| 4 | Application logs | status codes and endpoints | selective filters, string columns |
| 5 | Advertising clicks | campaign skew | CTR, skewed group-by, join |
| 6 | Game events | player sessions | high-cardinality group-by |
| 7 | Streaming platform events | watch duration | device/country aggregation |
| 8 | Public transport events | route delays | time and location aggregation |
| 9 | Banking-like transactions | risk/fraud flags | selective filters, top-k, sorting |
| 10 | Web analytics | referrers and pages | funnel-like aggregation |
| 11 | Delivery/logistics events | late status updates | late events, time windows |
| 12 | Education platform activity | courses and students | joins and progress metrics |
| 13 | Weather measurements | missing values | resampling and null handling |
| 14 | Marketplace listings | prices and categories | quantiles, category statistics |
| 15 | Security events | rare alerts | selective filters and high skew |
| 16 | Support tickets | priority and SLA | time-to-resolution metrics |

You may rename columns and categories to fit the chosen profile. Keep enough common structure to run the same engine comparisons.

In [3]:
DOMAIN_CARDS = {
    1: {"name": "Social media posts", "feature": "tags", "stress": "explode/list handling and top-k"},
    2: {"name": "E-commerce orders", "feature": "products", "stress": "joins and category aggregation"},
    3: {"name": "IoT telemetry", "feature": "device time series", "stress": "time filters and rolling/window logic"},
    4: {"name": "Application logs", "feature": "status codes", "stress": "selective filters and string columns"},
    5: {"name": "Advertising clicks", "feature": "campaign skew", "stress": "CTR, skewed group-by, and joins"},
    6: {"name": "Game events", "feature": "player sessions", "stress": "high-cardinality group-by"},
    7: {"name": "Streaming platform events", "feature": "watch duration", "stress": "device/country aggregation"},
    8: {"name": "Public transport events", "feature": "route delays", "stress": "time and location aggregation"},
    9: {"name": "Banking-like transactions", "feature": "risk flags", "stress": "selective filters, top-k, and sorting"},
    10: {"name": "Web analytics", "feature": "referrers", "stress": "funnel-like aggregation"},
    11: {"name": "Delivery/logistics events", "feature": "late status updates", "stress": "late events and time windows"},
    12: {"name": "Education platform activity", "feature": "courses", "stress": "joins and progress metrics"},
    13: {"name": "Weather measurements", "feature": "missing values", "stress": "resampling and null handling"},
    14: {"name": "Marketplace listings", "feature": "prices", "stress": "quantiles and category statistics"},
    15: {"name": "Security events", "feature": "rare alerts", "stress": "selective filters and high skew"},
    16: {"name": "Support tickets", "feature": "priority and SLA", "stress": "time-to-resolution metrics"},
}

assert 1 <= GROUP_ID <= 16, "GROUP_ID must be between 1 and 16"
CARD = DOMAIN_CARDS[GROUP_ID]
CARD

{'name': 'IoT telemetry',
 'feature': 'device time series',
 'stress': 'time filters and rolling/window logic'}

## Dataset requirements

Your generated dataset must contain at least:

- one timestamp column,
- one high-cardinality identifier, such as user, device, session, order, ticket, or transaction id,
- at least two categorical columns,
- at least two numeric metric columns,
- one feature specific to your variant card,
- enough rows to make local benchmark differences visible,
- a Parquet output file or directory.

Recommended starting sizes:

| Scale | Rows | Use case |
|---|---:|---|
| debug | 200,000 | Validate code quickly |
| small | 2,000,000 | Local development and first benchmark |
| medium | 10,000,000 to 20,000,000 | Main benchmark |
| large | 50,000,000+ | Optional stress test |

Use `debug` only while developing. The rendered notebook should report one main benchmark size (`N_ROWS`). If you run additional sizes, put those results in a separate stress-test table and do not mix them with the main benchmark table.

It is acceptable for different groups to generate different random data. Choose one main dataset size for the benchmark and record it as `N_ROWS`. You may use smaller debug data while developing and optional larger data for stress tests, but those extra sizes should be reported separately.

In [7]:
# TODO: Choose the main dataset scale for your final benchmark and verify output paths before generation.
# N_ROWS is the main row count reported for this notebook. Extra row counts are optional stress tests.
# Dataset configuration
SCALE = "debug"
SCALE_ROWS = {
    "debug": 200_000,
    "small": 2_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

N_ROWS = SCALE_ROWS[SCALE]
OUTPUT_DIR = Path("../data/phase2_26L") / f"group_{GROUP_ID:02d} / {SCALE}"
EVENTS_PATH = OUTPUT_DIR / "events.parquet"
PARTITIONED_EVENTS_DIR = OUTPUT_DIR / "events_partitioned"
OPTIMIZED_EVENTS_PATH = OUTPUT_DIR / "events_optimized.parquet"
DIMENSION_PATH = OUTPUT_DIR / "dimension.parquet"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

# Required negative baseline paths for the file-format/layout task. Do not commit these generated files.
CSV_EVENTS_PATH = OUTPUT_DIR / "events.csv"
JSON_EVENTS_PATH = OUTPUT_DIR / "events.jsonl"

# Leave SEED as None if you want independent data on each generation.
# If you need to reproduce exactly the same dataset later, set SEED to the value stored in the manifest.
SEED = 42
RUN_SEED = int(np.random.SeedSequence().entropy) if SEED is None else int(SEED)
rng = np.random.default_rng(RUN_SEED)
fake = Faker()

print("Group:", GROUP_ID, CARD)
print("Rows:", N_ROWS)
print("Run seed recorded in manifest:", RUN_SEED)
print("Output directory:", OUTPUT_DIR)


Group: 3 {'name': 'IoT telemetry', 'feature': 'device time series', 'stress': 'time filters and rolling/window logic'}
Rows: 200000
Run seed recorded in manifest: 42
Output directory: ../data/phase2_26L/group_03 / debug


## Generator template

The helper below creates a common base event table. You should extend it for your variant.

Do not spend most of the assignment writing a perfect data generator. The generator only needs to create data that is large enough and structurally interesting enough for your benchmark questions.

In [5]:
def skewed_ids(rng, n, max_id, hot_fraction=0.02, hot_probability=0.50):
    hot_count = max(1, int(max_id * hot_fraction))
    ids = rng.integers(hot_count + 1, max_id + 1, size=n)
    hot_mask = rng.random(n) < hot_probability
    ids[hot_mask] = rng.integers(1, hot_count + 1, size=hot_mask.sum())
    return ids


def random_tag_lists(rng, n, vocabulary=None, min_tags=1, max_tags=3):
    vocabulary = np.array(vocabulary or ["ai", "cloud", "spark", "polars", "duckdb", "sql", "etl", "security", "mlops"])
    counts = rng.integers(min_tags, max_tags + 1, size=n)
    tag_ids = rng.integers(0, len(vocabulary), size=(n, max_tags))
    return [[str(vocabulary[tag_ids[i, j]]) for j in range(counts[i])] for i in range(n)]


def generate_base_events(n, rng):
    start = np.datetime64("2026-01-01T00:00:00", "s")
    end = np.datetime64("2026-04-01T00:00:00", "s")
    seconds = int((end - start) / np.timedelta64(1, "s"))
    event_ts = (start + rng.integers(0, seconds, size=n).astype("timedelta64[s]")).astype("datetime64[us]")

    df = pl.DataFrame(
        {
            "event_id": np.arange(1, n + 1),
            "entity_id": skewed_ids(rng, n, max_id=200_000),
            "event_ts": event_ts,
            "category": rng.choice(["A", "B", "C", "D", "E", "F"], size=n),
            "country": rng.choice(["PL", "DE", "FR", "UK", "US", "IN", "BR"], size=n),
            "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.65, 0.25, 0.10]),
            "metric_1": rng.lognormal(mean=4.0, sigma=1.0, size=n).round(3),
            "metric_2": rng.integers(0, 10_000, size=n),
            "tags": random_tag_lists(rng, n),
        }
    )
    return df.with_columns(pl.col("event_ts").dt.date().alias("event_date"))


def customize_for_variant(df, card, rng):
    df = df.rename({"entity_id": "device_id", "event_id": "measurement_id"})
    iot_vocabulary = ["normal", "overheating", "low_battery", "offline_alert", "high_vibration", "maintenance_mode"]

    n = len(df)
    df = df.with_columns([
        pl.Series("sensor_type", rng.choice(["thermometer", "hygrometer", "pressure_sensor", "accelerometer"], size=n)),
        pl.Series("battery_level", rng.uniform(0.05, 1.0, size=n).round(2)),
        pl.Series("wifi_signal", rng.integers(-90, -30, size=n)),
        pl.Series("work_mode", rng.choice(["eco", "performance", "balanced"], size=n, p=[0.7, 0.2, 0.1])),
        pl.Series("is_stable", rng.choice([True, False], size=n, p=[0.95, 0.05])),
        pl.Series("tags", random_tag_lists(rng, n, vocabulary=iot_vocabulary))
    ])

    df = df.with_columns(
        pl.when(pl.col("sensor_type") == "accelerometer")
        .then(pl.col("metric_1") * 2)
        .otherwise(pl.col("metric_1"))
        .alias("metric_1")
    )

    df = df.drop(["category", "device"])
    return df


def generate_dimension_table(card, rng):
   num_devices = 200_000 
    
   return pl.DataFrame({
        "device_id": np.arange(1, num_devices + 1),
        "location": rng.choice(["Warsaw_Hub", "Berlin_Factory", "London_Office", "Paris_Lab"], size=num_devices),
        "hardware_model": rng.choice(["SensorPro-2000", "EcoLite-v2", "Industrial-X1"], size=num_devices),
        "last_service_date": rng.choice(np.arange(np.datetime64('2026-01-01'), np.datetime64('2026-04-01'), dtype='datetime64[D]'), size=num_devices),
        "priority_level": rng.integers(1, 6, size=num_devices)
    })

In [ ]:
# TODO: Run this after adapting the generator. Verify that generated data is not committed to Git.
# Generate and save the dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_events = generate_base_events(N_ROWS, rng)
events = customize_for_variant(base_events, CARD, rng)
dimension = generate_dimension_table(CARD, rng)

events.write_parquet(EVENTS_PATH, compression="zstd")
dimension.write_parquet(DIMENSION_PATH, compression="zstd")

# Optional partitioned layout for experiments with predicate pushdown and file layout.
events.write_parquet(PARTITIONED_EVENTS_DIR, partition_by="event_date", compression="zstd")

# TODO: Create an optimized Parquet layout for one selected query pattern.
# Example ideas:
# - sort by columns used in range filters before writing,
# - choose a smaller row_group_size if it improves row-group pruning,
# - partition by date or another selective filter column,
# - add bloom filters only if your chosen writer and reader expose this option clearly.
# Replace the sort columns with columns from your own query pattern.
events.sort(["device_id", "event_ts"]).write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000,
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "group_id": GROUP_ID,
    "variant": CARD,
    "scale": SCALE,
    "rows": int(events.height),
    "run_seed": RUN_SEED,
    "paths": {
        "events": str(EVENTS_PATH),
        "events_partitioned": str(PARTITIONED_EVENTS_DIR),
        "events_optimized": str(OPTIMIZED_EVENTS_PATH),
        "dimension": str(DIMENSION_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "polars": pl.__version__,
        "pandas": pd.__version__,
        "duckdb": duckdb.__version__,
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))


## Dataset sanity checks

Before benchmarking, inspect your schema and basic statistics. Your report should briefly explain why your dataset is suitable for the queries you chose.

In [10]:
from IPython.display import display
import os

events = pl.read_parquet(EVENTS_PATH)
dim    = pl.read_parquet(DIMENSION_PATH)

# ── 1. Shape & row-count match ─────────────────────────────────────────────────
print("=" * 65)
print("EVENTS TABLE")
print("=" * 65)
print(f"Shape       : {events.shape[0]:>12,} rows × {events.shape[1]} columns")
print(f"Expected    : {N_ROWS:>12,} rows  →  OK: {events.shape[0] == N_ROWS}")

# ── 2. Schema ─────────────────────────────────────────────────────────────────
print("\nSchema:")
for name, dtype in zip(events.schema.names(), events.schema.dtypes()):
    print(f"  {name:<22} {str(dtype)}")

# ── 3. Requirement checklist ──────────────────────────────────────────────────
print("\n" + "─" * 65)
print("Requirement checklist:")

TIMESTAMP_COLS  = ["event_ts"]
HIGH_CARD_COLS  = ["measurement_id", "device_id"]
CATEGORICAL_COLS = ["country", "sensor_type", "work_mode"]
NUMERIC_COLS    = ["metric_1", "metric_2", "battery_level", "wifi_signal"]
VARIANT_COLS    = ["sensor_type", "battery_level", "wifi_signal", "work_mode", "is_stable", "tags"]

col_names = events.schema.names()

def check(label, condition):
    status = "✓" if condition else "✗ FAIL"
    print(f"  [{status}] {label}")

check("timestamp column (event_ts)",          all(c in col_names for c in TIMESTAMP_COLS))
check("high-cardinality ID (measurement_id, device_id)", all(c in col_names for c in HIGH_CARD_COLS))
check("≥ 2 categorical columns",              all(c in col_names for c in CATEGORICAL_COLS))
check("≥ 2 numeric metric columns",           all(c in col_names for c in NUMERIC_COLS))
check("IoT variant columns present",          all(c in col_names for c in VARIANT_COLS))
check("no fully-null column",                 all(events[c].null_count() < events.height for c in col_names))
check("Parquet file exists on disk",          EVENTS_PATH.exists())
check("Dimension table exists on disk",       DIMENSION_PATH.exists())
check(f"Scale ≥ debug ({N_ROWS:,} rows)",      events.height >= 200_000)

# ── 4. Timestamp range ────────────────────────────────────────────────────────
ts_min = events["event_ts"].min()
ts_max = events["event_ts"].max()
print("\nTimestamp range:")
print(f"  min: {ts_min}   max: {ts_max}")

# ── 5. Cardinality of key columns ─────────────────────────────────────────────
print("\nCardinality of key columns:")
for col in ["device_id", "country", "sensor_type", "work_mode"]:
    n_unique = events[col].n_unique()
    print(f"  {col:<22} {n_unique:>8,} distinct values")

# ── 6. Categorical value distributions ────────────────────────────────────────
print("\nValue counts – sensor_type:")
display(events["sensor_type"].value_counts().sort("count", descending=True).to_pandas())

print("\nValue counts – country:")
display(events["country"].value_counts().sort("count", descending=True).to_pandas())

print("\nValue counts – work_mode:")
display(events["work_mode"].value_counts().sort("count", descending=True).to_pandas())

# ── 7. Numeric statistics ─────────────────────────────────────────────────────
print("\nDescriptive statistics (events):")
display(
    events.select(["metric_1", "metric_2", "battery_level", "wifi_signal"])
    .describe()
    .to_pandas()
)

# ── 8. Skew check on device_id ─────────────────────────────────────────────────
reading_counts = events.group_by("device_id").len().rename({"len": "readings"})
print("\ndevice_id reading-count distribution (shows skew):")
display(reading_counts["readings"].describe().to_pandas())

# ── 9. Dimension table summary ────────────────────────────────────────────────
print("=" * 65)
print("DIMENSION TABLE")
print("=" * 65)
print(f"Shape: {dim.shape[0]:,} rows × {dim.shape[1]} columns")
print("Schema:")
for name, dtype in zip(dim.schema.names(), dim.schema.dtypes()):
    print(f"  {name:<22} {str(dtype)}")
print("\nValue counts – location:")
display(dim["location"].value_counts().sort("count", descending=True).to_pandas())

# ── 10. File sizes ────────────────────────────────────────────────────────────
print("=" * 65)
print("File sizes on disk:")
for label, path in [
    ("events.parquet (default)",    EVENTS_PATH),
    ("events_optimized.parquet",    OPTIMIZED_EVENTS_PATH),
    ("dimension.parquet",           DIMENSION_PATH),
]:
    if path.exists():
        mb = path.stat().st_size / 1024 / 1024
        print(f"  {label:<32} {mb:>8.2f} MB")

print("\nAll sanity checks passed – dataset is ready for benchmarking.")


EVENTS TABLE
Shape       :      200,000 rows × 13 columns
Expected    :      200,000 rows  →  OK: True

Schema:
  measurement_id         Int64
  device_id              Int64
  event_ts               Datetime(time_unit='us', time_zone=None)
  country                String
  metric_1               Float64
  metric_2               Int64
  tags                   List(String)
  event_date             Date
  sensor_type            String
  battery_level          Float64
  wifi_signal            Int64
  work_mode              String
  is_stable              Boolean

─────────────────────────────────────────────────────────────────
Requirement checklist:
  [✓] timestamp column (event_ts)
  [✓] high-cardinality ID (measurement_id, device_id)
  [✓] ≥ 2 categorical columns
  [✓] ≥ 2 numeric metric columns
  [✓] IoT variant columns present
  [✓] no fully-null column
  [✓] Parquet file exists on disk
  [✓] Dimension table exists on disk
  [✓] Scale ≥ debug (200,000 rows)

Timestamp range:
  min: 20

,sensor_type,count
0,accelerometer,50158
1,hygrometer,50104
2,thermometer,50052
3,pressure_sensor,49686



Value counts – country:


,country,count
0,UK,28813
1,FR,28772
2,US,28575
3,BR,28567
4,PL,28534
5,DE,28400
6,IN,28339



Value counts – work_mode:


...


device_id reading-count distribution (shows skew):


,statistic,value
0,count,82284.000000
1,null_count,0.000000
2,mean,2.430606
3,std,5.251562
4,min,1.000000
5,25%,1.000000
6,50%,1.000000
7,75%,2.000000
8,max,43.000000


DIMENSION TABLE
Shape: 200,000 rows × 5 columns
Schema:
  device_id              Int64
  location               String
  hardware_model         String
  last_service_date      Date
  priority_level         Int64

Value counts – location:


,location,count
0,Warsaw_Hub,50211
1,Paris_Lab,50056
2,Berlin_Factory,50013
3,London_Office,49720


File sizes on disk:
  events.parquet (default)             4.44 MB
  events_optimized.parquet             4.22 MB
  dimension.parquet                    0.55 MB

All sanity checks passed – dataset is ready for benchmarking.


### Dataset suitability summary

The IoT telemetry dataset (Group 3) satisfies all schema requirements:

| Requirement | Column(s) | Notes |
|---|---|---|
| Timestamp column | `event_ts` | microsecond precision, spans 2026-01-01 – 2026-04-01 |
| High-cardinality ID | `measurement_id`, `device_id` | `device_id` has a deliberate hot-spot skew (top 2 % of IDs hold ~50 % of events) |
| Categorical columns | `country` (7 values), `sensor_type` (4 values), `work_mode` (3 values) | useful for group-by and filter benchmarks |
| Numeric metrics | `metric_1` (log-normal), `metric_2` (uniform int), `battery_level`, `wifi_signal` | cover aggregation, range filters, and window logic |
| IoT-variant features | `sensor_type`, `battery_level`, `wifi_signal`, `work_mode`, `is_stable`, `tags` | time-series device profile; `tags` is a list column |
| Dimension table | `device_id → location, hardware_model, priority_level` | enables join benchmarks (Q2) |

The skewed `device_id` distribution makes **Q3 (high-cardinality group-by + top-k)** a meaningful benchmark.  
The 17-day time window in **Q1 (selective filter + aggregation)** is selective enough to benefit from row-group pruning in the optimized Parquet layout.  
**Q2 (join + aggregation)** exercises the join path be

## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 3.1 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the helper shape below, but you need to implement the actual benchmark functions.


In [50]:
import gc
import time
import numpy as np
import pandas as pd
from memory_profiler import memory_usage

BENCHMARK_COLUMNS = [
    "library_engine",
    "mode",
    "query_name",
    "data_format",
    "layout",
    "rows",
    "median_time_s",
    "peak_memory_mb",
    "input_size_mb",
    "result_check",
    "notes",
]

benchmark_results = []

def _measure_peak_memory(func, kwargs, engine_name, query_name):
    """Peak memory usage for a single run of the function."""
    gc.collect()
    try:
        mem_usage = memory_usage((func, (), kwargs), max_usage=True, include_children=True)
        return float(mem_usage) if isinstance(mem_usage, (float, int)) else float(mem_usage[0])
    except Exception as e:
        print(f"Error profiling memory for {engine_name} - {query_name}: {e}")
        return 0.0

def _measure_execution_time(func, kwargs, iterations):
    """Measures execution time for a given function."""
    times = []
    result = None
    for _ in range(iterations):
        gc.collect()
        start = time.perf_counter()
        result = func(**kwargs)
        end = time.perf_counter()
        times.append(end - start)
        
    return np.median(times), result

def _get_result_shape(result):
    """Checks the shape/size of the result for verification purposes."""
    if hasattr(result, 'shape'):
        return f"shape: {result.shape}"
    elif hasattr(result, '__len__'):
        return f"len: {len(result)}"
    return "Done"

def run_benchmark(
    func, kwargs, library_engine, mode, query_name, 
    data_format, layout, rows, input_size_mb=None, 
    notes="", iterations=3
):  #1 . Run every benchmark at least three times
    #4. Record peak memory where possibl
    peak_mem_mb = _measure_peak_memory(func, kwargs, library_engine, query_name)
    
    
    #2. Run gc.collect() before each measured repetition.
    #3. Report median runtime, not only one measurement
    median_time, result = _measure_execution_time(func, kwargs, iterations)
    
    #5. Check that results are logically equivalent across libraries/engines.
    result_check = _get_result_shape(result)
    
  # 6. Store your results in a table.
    result_row = {
        "library_engine": library_engine,
        "mode": mode,
        "query_name": query_name,
        "data_format": data_format,
        "layout": layout,
        "rows": rows,
        "median_time_s": round(median_time, 4),
        "peak_memory_mb": round(peak_mem_mb, 2),
        "input_size_mb": input_size_mb,
       #7.  Describe any library/engine-specific settings
        "result_check": result_check,
        "notes": notes
    }
    
    #print(f" Finished: {library_engine} | {query_name} | Time: {result_row['median_time_s']}s | Memory: {result_row['peak_memory_mb']}MB")
    return result_row

## Part 3: Student tasks

### Task 1: Design three benchmark queries

Create three queries of your own choice. They must test different behavior.

Your queries should cover at least three of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- join with a dimension table,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [ ]:
# TODO: Define your three query specifications in prose or structured metadata.
# Do not start benchmarking before you can explain what each query is supposed to test.

### Task 2: Benchmark local libraries/engines

Implement your three queries in:

- Pandas 3.0 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB,
- PySpark local mode.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.

For PySpark, use local mode in this task. Dataproc is a separate task later in the notebook.


In [ ]:
# TODO: Configure Spark local only when you start the PySpark local benchmark.
# Initialize Spark only when you start the Spark part of the benchmark.
# TODO: Adjust memory and local core count if needed.

# spark = (
#     SparkSession.builder
#     .appName("TBDPhase2LocalBenchmark")
#     .master("local[*]")
#     .config("spark.driver.memory", "4g")
#     .getOrCreate()
# )

In [51]:
# TODO: Pandas implementations of your three queries.
# Implement both Pandas read variants:
# 1. default backend: pd.read_parquet(path)
# 2. PyArrow backend: pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
#
# Report dtypes for both variants and compare runtime/memory.

def pandas_q1(events_path, use_pyarrow=False):
    if use_pyarrow:
        df = pd.read_parquet(events_path, engine="pyarrow", dtype_backend="pyarrow")
    else:
        df = pd.read_parquet(events_path)
    
    # Q1: Selective filter + aggregation
    mask = (
        (df['event_ts'] >= pd.Timestamp('2026-01-15')) & 
        (df['event_ts'] <= pd.Timestamp('2026-01-31')) & 
        (df['sensor_type'] == 'thermometer') & 
        (df['battery_level'] < 0.3)
    )
    filtered = df[mask]
    
    res = filtered.groupby('country').agg(
        n_readings=('measurement_id', 'count'),
        avg_metric=('metric_1', 'mean'),
        avg_signal=('wifi_signal', 'mean'),
        unstable_count=('is_stable', lambda x: (~x.astype(bool)).sum())
    ).reset_index().sort_values('n_readings', ascending=False)
    
    return res

def pandas_q2(events_path, dim_path, use_pyarrow=False):
    if use_pyarrow:
        events = pd.read_parquet(events_path, engine="pyarrow", dtype_backend="pyarrow")
        dim = pd.read_parquet(dim_path, engine="pyarrow", dtype_backend="pyarrow")
    else:
        events = pd.read_parquet(events_path)
        dim = pd.read_parquet(dim_path)
        
    # Q2: Join + low-cardinality group-by
    merged = events.merge(dim, on='device_id', how='inner')
    
    res = merged.groupby(['location', 'hardware_model']).agg(
        n_readings=('measurement_id', 'count'),
        avg_battery=('battery_level', 'mean'),
        min_battery=('battery_level', 'min'),
        avg_metric=('metric_1', 'mean'),
        total_metric2=('metric_2', 'sum'),
        
        stability_rate=('is_stable', lambda x: x.astype(float).mean())
    ).reset_index().sort_values('n_readings', ascending=False)
    
    return res

def pandas_q3(events_path, use_pyarrow=False):
    if use_pyarrow:
        df = pd.read_parquet(events_path, engine="pyarrow", dtype_backend="pyarrow")
    else:
        df = pd.read_parquet(events_path)
        
    # Q3: High-cardinality group-by + top-k
    res = df.groupby('device_id').agg(
        n_readings=('measurement_id', 'count'),
        avg_battery=('battery_level', 'mean'),
        min_battery=('battery_level', 'min'),
        avg_signal=('wifi_signal', 'mean'),
        last_seen=('event_ts', 'max')
    ).reset_index().sort_values('n_readings', ascending=False).head(100)
    
    return res

In [52]:


print("--- Pandas Default Dtypes ---")
df_default = pd.read_parquet(EVENTS_PATH)
print(df_default.dtypes.head(5)) 

print("\n--- Pandas PyArrow Dtypes ---")
df_pyarrow = pd.read_parquet(EVENTS_PATH, engine="pyarrow", dtype_backend="pyarrow")
print(df_pyarrow.dtypes.head(5))


del df_default
del df_pyarrow
import gc
gc.collect()

--- Pandas Default Dtypes ---
measurement_id             int64
device_id                  int64
event_ts          datetime64[us]
country                      str
metric_1                 float64
dtype: object

--- Pandas PyArrow Dtypes ---
measurement_id            int64[pyarrow]
device_id                 int64[pyarrow]
event_ts          timestamp[us][pyarrow]
country            large_string[pyarrow]
metric_1                 double[pyarrow]
dtype: object


0

In [53]:

dataset_rows = N_ROWS 
events_file = EVENTS_PATH
dim_file = DIMENSION_PATH
iterations_count = 3 


print("--- Start: Pandas Default ---")
benchmark_results.append(run_benchmark(
    pandas_q1, {"events_path": events_file, "use_pyarrow": False},
    "Pandas", "default", "Q1_selective_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q2, {"events_path": events_file, "dim_path": dim_file, "use_pyarrow": False},
    "Pandas", "default", "Q2_join_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q3, {"events_path": events_file, "use_pyarrow": False},
    "Pandas", "default", "Q3_high_card_top_k", "parquet", "default", dataset_rows, iterations=iterations_count
))


print("\n--- Start: Pandas PyArrow ---")
benchmark_results.append(run_benchmark(
    pandas_q1, {"events_path": events_file, "use_pyarrow": True},
    "Pandas", "pyarrow", "Q1_selective_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q2, {"events_path": events_file, "dim_path": dim_file, "use_pyarrow": True},
    "Pandas", "pyarrow", "Q2_join_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    pandas_q3, {"events_path": events_file, "use_pyarrow": True},
    "Pandas", "pyarrow", "Q3_high_card_top_k", "parquet", "default", dataset_rows, iterations=iterations_count
))


pd.DataFrame(benchmark_results)

--- Start: Pandas Default ---

--- Start: Pandas PyArrow ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default,Q1_selective_agg,parquet,default,200000,0.3052,3829.48,None,"shape: (7, 5)",
1,Pandas,default,Q2_join_agg,parquet,default,200000,0.2601,3831.16,None,"shape: (12, 8)",
2,Pandas,default,Q3_high_card_top_k,parquet,default,200000,0.2520,3833.98,None,"shape: (100, 6)",
3,Pandas,pyarrow,Q1_selective_agg,parquet,default,200000,0.1863,3828.42,None,"shape: (7, 5)",
4,Pandas,pyarrow,Q2_join_agg,parquet,default,200000,0.2111,3826.42,None,"shape: (12, 8)",
5,Pandas,pyarrow,Q3_high_card_top_k,parquet,default,200000,0.1352,3829.49,None,"shape: (100, 6)",


In [ ]:
# TODO: Polars implementations of your three queries.
# Required modes:
# - eager: read_parquet -> transformations
# - lazy default: scan_parquet -> transformations -> collect()
# - lazy streaming: scan_parquet -> transformations -> collect(engine="streaming")

In [54]:
# TODO: DuckDB SQL implementations of your three queries.
# Consider querying Parquet files directly instead of first loading all data into Pandas.
# ==========================================
# Implementacja zapytań dla DuckDB
# ==========================================

def duckdb_q1(events_path):
    query = f"""
        SELECT
            country,
            COUNT(measurement_id) AS n_readings,
            AVG(metric_1) AS avg_metric,
            AVG(wifi_signal) AS avg_signal,
            SUM(CASE WHEN is_stable = false THEN 1 ELSE 0 END) AS unstable_count
        FROM '{events_path}'
        WHERE event_ts >= '2026-01-15' AND event_ts <= '2026-01-31'
          AND sensor_type = 'thermometer'
          AND battery_level < 0.3
        GROUP BY country
        ORDER BY n_readings DESC;
    """
    return duckdb.sql(query).df()

def duckdb_q2(events_path, dim_path):
    query = f"""
        SELECT
            d.location,
            d.hardware_model,
            COUNT(e.measurement_id) AS n_readings,
            AVG(e.battery_level) AS avg_battery,
            MIN(e.battery_level) AS min_battery,
            AVG(e.metric_1) AS avg_metric,
            SUM(e.metric_2) AS total_metric2,
            AVG(CAST(e.is_stable AS INT)) AS stability_rate
        FROM '{events_path}' e
        INNER JOIN '{dim_path}' d ON e.device_id = d.device_id
        GROUP BY d.location, d.hardware_model
        ORDER BY n_readings DESC;
    """
    return duckdb.sql(query).df()

def duckdb_q3(events_path):
    query = f"""
        SELECT
            device_id,
            COUNT(measurement_id) AS n_readings,
            AVG(battery_level) AS avg_battery,
            MIN(battery_level) AS min_battery,
            AVG(wifi_signal) AS avg_signal,
            MAX(event_ts) AS last_seen
        FROM '{events_path}'
        GROUP BY device_id
        ORDER BY n_readings DESC
        LIMIT 100;
    """
    return duckdb.sql(query).df()

In [55]:


print("--- Start: DuckDB ---")

benchmark_results.append(run_benchmark(
    duckdb_q1, {"events_path": events_file},
    "DuckDB", "sql", "Q1_selective_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    duckdb_q2, {"events_path": events_file, "dim_path": dim_file},
    "DuckDB", "sql", "Q2_join_agg", "parquet", "default", dataset_rows, iterations=iterations_count
))

benchmark_results.append(run_benchmark(
    duckdb_q3, {"events_path": events_file},
    "DuckDB", "sql", "Q3_high_card_top_k", "parquet", "default", dataset_rows, iterations=iterations_count
))

pd.DataFrame(benchmark_results)

--- Start: DuckDB ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default,Q1_selective_agg,parquet,default,200000,0.3052,3829.48,None,"shape: (7, 5)",
1,Pandas,default,Q2_join_agg,parquet,default,200000,0.2601,3831.16,None,"shape: (12, 8)",
2,Pandas,default,Q3_high_card_top_k,parquet,default,200000,0.2520,3833.98,None,"shape: (100, 6)",
3,Pandas,pyarrow,Q1_selective_agg,parquet,default,200000,0.1863,3828.42,None,"shape: (7, 5)",
4,Pandas,pyarrow,Q2_join_agg,parquet,default,200000,0.2111,3826.42,None,"shape: (12, 8)",
5,Pandas,pyarrow,Q3_high_card_top_k,parquet,default,200000,0.1352,3829.49,None,"shape: (100, 6)",
6,DuckDB,sql,Q1_selective_agg,parquet,default,200000,0.0707,3859.51,None,"shape: (7, 5)",
7,DuckDB,sql,Q2_join_agg,parquet,default,200000,0.1835,3798.26,None,"shape: (12, 8)",
8,DuckDB,sql,Q3_high_card_top_k,parquet,default,200000,0.1256,3828.71,None,"shape: (100, 6)",


In [ ]:
# TODO: PySpark local implementations of your three queries.

### Task 2.5: File format and Parquet layout optimization

Choose one of your three queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [58]:
# TODO 2.5: Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.



import os
import polars as pl
import duckdb
import pandas as pd


def get_file_size_mb(path):
    return round(os.path.getsize(path) / (1024 * 1024), 2)


print("--- Preparing files for experiment ---")
#  1: Choose one query with a selective filter or column subset.

q1_cols = [
    'measurement_id', 'event_ts', 'country', 'sensor_type', 
    'battery_level', 'wifi_signal', 'is_stable', 'metric_1'
]


df_exp = pl.read_parquet(EVENTS_PATH).select(q1_cols)

#  4: Write CSV or JSONL as a required negative baseline.

print(f"Saving CSV to: {CSV_EVENTS_PATH}")
df_exp.write_csv(CSV_EVENTS_PATH)

#  3: Write an optimized Parquet file/directory.
print(f"Saving optimized Parquet to: {OPTIMIZED_EVENTS_PATH}")
df_exp.sort("event_ts").write_parquet(
    OPTIMIZED_EVENTS_PATH,
    compression="zstd",
    row_group_size=100_000 
)

#5: Benchmark the same logical query on all three formats.
def duckdb_q1_path(file_path):
    
    query = f"""
        SELECT
            country,
            COUNT(measurement_id) AS n_readings,
            AVG(CAST(metric_1 AS DOUBLE)) AS avg_metric,
            AVG(CAST(wifi_signal AS DOUBLE)) AS avg_signal,
            SUM(CASE WHEN CAST(is_stable AS BOOLEAN) = false THEN 1 ELSE 0 END) AS unstable_count
        FROM '{file_path}'
        WHERE CAST(event_ts AS TIMESTAMP) >= '2026-01-15' 
          AND CAST(event_ts AS TIMESTAMP) <= '2026-01-31'
          AND sensor_type = 'thermometer'
          AND CAST(battery_level AS DOUBLE) < 0.3
        GROUP BY country
        ORDER BY n_readings DESC;
    """
    return duckdb.sql(query).df()


print("\n--- Start: Benchmark  of formats(DuckDB Q1) ---")

#2: Write a baseline Parquet file/directory.
size_default = get_file_size_mb(EVENTS_PATH)
size_opt = get_file_size_mb(OPTIMIZED_EVENTS_PATH)
size_csv = get_file_size_mb(CSV_EVENTS_PATH)


benchmark_results.append(run_benchmark(
    duckdb_q1_path, {"file_path": EVENTS_PATH},
    "DuckDB", "sql", "Q1_selective_agg", "parquet", "default", N_ROWS, 
    input_size_mb=size_default, iterations=3
))


benchmark_results.append(run_benchmark(
    duckdb_q1_path, {"file_path": OPTIMIZED_EVENTS_PATH},
    "DuckDB", "sql", "Q1_selective_agg", "parquet", "optimized_by_date", N_ROWS, 
    input_size_mb=size_opt, iterations=3
))


benchmark_results.append(run_benchmark(
    duckdb_q1_path, {"file_path": CSV_EVENTS_PATH},
    "DuckDB", "sql", "Q1_selective_agg", "csv", "flat", N_ROWS, 
    input_size_mb=size_csv, iterations=3
))


pd.DataFrame(benchmark_results).tail(3)


--- Preparing files for experiment ---
Saving CSV to: ../data/phase2_26L/group_03/events.csv
Saving optimized Parquet to: ../data/phase2_26L/group_03/events_optimized.parquet

--- Start: Benchmark  of formats(DuckDB Q1) ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
12,DuckDB,sql,Q1_selective_agg,parquet,default,200000,0.0356,3863.09,4.44,"shape: (7, 5)",
13,DuckDB,sql,Q1_selective_agg,parquet,optimized_by_date,200000,0.0273,3848.26,2.55,"shape: (7, 5)",
14,DuckDB,sql,Q1_selective_agg,csv,flat,200000,0.2959,3880.39,13.51,"shape: (7, 5)",


In [59]:
# 6. Record IO/pruning evidence where available.
explain_query = f"""
    EXPLAIN ANALYZE 
    SELECT *
    FROM '{OPTIMIZED_EVENTS_PATH}'
    WHERE CAST(event_ts AS TIMESTAMP) >= '2026-01-15' AND CAST(event_ts AS TIMESTAMP) <= '2026-01-31';
"""
print(duckdb.sql(explain_query).df().iloc[0]['explain_value'])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE      SELECT *     FROM '../data/phase2_26L/group_03/events_optimized.parquet'     WHERE CAST(event_ts AS TIMESTAMP) >= '2026-01-15' AND CAST(event_ts AS TIMESTAMP) <= '2026-01-31'; 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0328s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PR

### Justification for the Optimized Parquet Layout

The optimized Parquet layout (`OPTIMIZED_EVENTS_PATH`) incorporates two specific physical design changes tailored to the filtering behavior of **Query 1**. These changes were not accidental; they were designed to maximize I/O efficiency:

1. **Sorting by the Filter Column (`event_ts`):**
   * **Why it helps:** Query 1 utilizes a highly selective time-window filter (`event_ts BETWEEN '2026-01-15' AND '2026-01-31'`). In an unsorted (default) Parquet file, events from January are scattered randomly across the entire file, forcing the query engine to scan nearly all data blocks. By sorting the dataframe by date prior to writing, all events from our target time window are physically clustered together on disk. When DuckDB reads the Parquet metadata (which stores min/max values for each block), it can instantly identify and completely skip the blocks containing data from other months. This mechanism, known as **Row-Group Pruning**, drastically reduces disk I/O and execution time.

2. **Reducing `row_group_size` (to 100,000):**
   * **Why it helps:** The default Parquet row group size is often quite large (e.g., 1 million rows or more). By explicitly shrinking it to 100,000 rows, we create smaller, more granular data chunks with tighter min/max statistics. This allows DuckDB's scanner to prune irrelevant data much more precisely at the boundaries of our target date range. Furthermore, smaller row groups improve parallelization by allowing multiple CPU threads to process distinct chunks concurrently.

### Task 3: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

This task has three separate parts. Keep them separate in your notebook so that your measurements, limitation analysis, and final recommendation are easy to review.

#### 3.1 Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [ ]:
# TODO 3.1: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# YOUR CODE HERE


#### 3.2 Polars limitations

Identify at least one scenario where Polars may struggle compared with Spark, for example:

- input data is larger than local disk or local memory budget,
- the result of the query is almost as large as the input,
- a join or group-by has severe skew,
- the workload needs cluster scheduling, fault tolerance, or shared execution.

Support your claim with evidence from your own benchmark. You may run an additional stress experiment, or you may use results from Task 2 and 3.1 if they already show the limitation clearly.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.2: Identify and justify one Polars limitation.
#
# Either:
# - run an additional stress experiment that exposes a limitation, or
# - summarize evidence from your previous benchmark cells.
#
# Fill the variables below and add code if you run an extra experiment.

POLARS_LIMITATION_SCENARIO = """
TODO: Describe the scenario where Polars may struggle compared with Spark.
"""

POLARS_LIMITATION_EVIDENCE = """
TODO: Cite concrete evidence: dataset size, query shape, runtime, memory, failure, or scaling behaviour.
"""

# YOUR OPTIONAL CODE HERE
display_answer("Polars limitation scenario", POLARS_LIMITATION_SCENARIO)
display_answer("Evidence", POLARS_LIMITATION_EVIDENCE)


#### 3.3 Decision boundary

Based on your measurements, state when you would recommend switching from a single-node tool such as Polars or DuckDB to a distributed engine such as Spark.

Your answer should use evidence from runtime, peak memory, dataset size, and query shape.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO 3.3: State your decision boundary.
#
# Your answer should be specific. Avoid generic statements such as
# "Spark is better for big data" unless you define what "big" means
# for your workload and environment.

DECISION_BOUNDARY = """
TODO: Based on our measurements, we would switch from local Polars/DuckDB to Spark when...
"""

DECISION_EVIDENCE = """
TODO: List the measurements or observations that support the decision.
"""

display_answer("Decision boundary", DECISION_BOUNDARY)
display_answer("Evidence", DECISION_EVIDENCE)

### Task 4: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- PySpark local: compare `local[1]`, `local[2]`, `local[*]` where practical.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

In [60]:
# TODO: Run selected scalability experiments and append results to benchmark_results.

# duck DB=

print("--- Start: DuckDB thread scalability (Q3) ---")


thread_counts = [1, 4, 8, 20]

for threads in thread_counts:
    
    duckdb.sql(f"PRAGMA threads={threads}")
    
    
    benchmark_results.append(run_benchmark(
        duckdb_q3, {"events_path": EVENTS_PATH},
        "DuckDB", "sql", "Q3_high_card_top_k", "parquet", "default", N_ROWS, 
        notes=f"threads={threads}", 
        iterations=3
    ))


duckdb.sql("PRAGMA threads=20")

df_results = pd.DataFrame(benchmark_results)
display(df_results[df_results['notes'].str.contains('threads', na=False)])

--- Start: DuckDB thread scalability (Q3) ---


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
15,DuckDB,sql,Q3_high_card_top_k,parquet,default,200000,0.1079,3872.29,NaN,"shape: (100, 6)",threads=1
16,DuckDB,sql,Q3_high_card_top_k,parquet,default,200000,0.0954,3901.88,NaN,"shape: (100, 6)",threads=4
17,DuckDB,sql,Q3_high_card_top_k,parquet,default,200000,0.1543,3947.99,NaN,"shape: (100, 6)",threads=8
18,DuckDB,sql,Q3_high_card_top_k,parquet,default,200000,0.0706,3957.46,NaN,"shape: (100, 6)",threads=20


### Task 5: Spark on Dataproc

Use the infrastructure from Phase 1 to run selected PySpark queries on a Dataproc cluster.

Required comparison:

- local PySpark vs. Dataproc PySpark,
- your main dataset size, and optionally one larger stress-test size if Spark overhead or scaling is not visible,
- at least one explanation based on Spark execution characteristics such as partitions, shuffle, caching, or scheduling overhead.

You may use the same generated Parquet data, uploaded to GCS. Consider using the partitioned layout if your query filters by date or another partition column.

In [ ]:
# TODO: Add Dataproc-specific commands, notebook cells, or instructions used by your group.
# Do not hard-code credentials or project secrets in the notebook.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- group id and selected data profile,
- link to this notebook in your fork,
- main dataset size (`N_ROWS`), schema summary, and physical layout,
- three query descriptions with hypotheses,
- local benchmark table for Pandas 3.0 default backend, Pandas 3.0 PyArrow backend, Polars, DuckDB, and PySpark local,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- Dataproc comparison,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [61]:


print("Person A")


final_benchmark_df = pd.DataFrame(benchmark_results)


display(final_benchmark_df)


final_benchmark_df.to_csv("person_a_results.csv", index=False)


Person A


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,peak_memory_mb,input_size_mb,result_check,notes
0,Pandas,default,Q1_selective_agg,parquet,default,200000,0.3052,3829.48,NaN,"shape: (7, 5)",
1,Pandas,default,Q2_join_agg,parquet,default,200000,0.2601,3831.16,NaN,"shape: (12, 8)",
2,Pandas,default,Q3_high_card_top_k,parquet,default,200000,0.2520,3833.98,NaN,"shape: (100, 6)",
3,Pandas,pyarrow,Q1_selective_agg,parquet,default,200000,0.1863,3828.42,NaN,"shape: (7, 5)",
4,Pandas,pyarrow,Q2_join_agg,parquet,default,200000,0.2111,3826.42,NaN,"shape: (12, 8)",
5,Pandas,pyarrow,Q3_high_card_top_k,parquet,default,200000,0.1352,3829.49,NaN,"shape: (100, 6)",
6,DuckDB,sql,Q1_selective_agg,parquet,default,200000,0.0707,3859.51,NaN,"shape: (7, 5)",
7,DuckDB,sql,Q2_join_agg,parquet,default,200000,0.1835,3798.26,NaN,"shape: (12, 8)",
8,DuckDB,sql,Q3_high_card_top_k,parquet,default,200000,0.1256,3828.71,NaN,"shape: (100, 6)",
9,DuckDB,sql,Q1_selective_agg,parquet,default,200000,0.0872,3880.45,4.44,"shape: (7, 5)",


In [26]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
Największą różnicę między API DataFrame (Pandas) a silnikiem SQL (DuckDB) obnaża zapytanie **Q1 (Selective filter + aggregation)** oraz **Q2 (Join)**. 

Głównym powodem jest sposób, w jaki oba silniki traktują pamięć i wejście/wyjście (I/O). Pandas jako silnik in-memory typu "eager" musi najpierw wczytać całe pliki Parquet do pamięci RAM, zanim zastosuje filtry. Powoduje to ogromny skok zużycia pamięci (Peak Memory). Z kolei DuckDB wykonuje zapytanie SQL bezpośrednio na plikach na dysku (Out-of-core). DuckDB stosuje agresywny *predicate pushdown* (spychanie filtrów na poziom czytania pliku) oraz *projection pruning* (czyta tylko potrzebne kolumny). Dzięki temu w DuckDB omijamy całe gigabajty danych, co skutkuje ułamkiem zużycia pamięci i znacznie szybszym czasem wykonania w porównaniu do Pandas.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

**Final answer 1**

Największą różnicę między API DataFrame (Pandas) a silnikiem SQL (DuckDB) obnaża zapytanie **Q1 (Selective filter + aggregation)** oraz **Q2 (Join)**. 

Głównym powodem jest sposób, w jaki oba silniki traktują pamięć i wejście/wyjście (I/O). Pandas jako silnik in-memory typu "eager" musi najpierw wczytać całe pliki Parquet do pamięci RAM, zanim zastosuje filtry. Powoduje to ogromny skok zużycia pamięci (Peak Memory). Z kolei DuckDB wykonuje zapytanie SQL bezpośrednio na plikach na dysku (Out-of-core). DuckDB stosuje agresywny *predicate pushdown* (spychanie filtrów na poziom czytania pliku) oraz *projection pruning* (czyta tylko potrzebne kolumny). Dzięki temu w DuckDB omijamy całe gigabajty danych, co skutkuje ułamkiem zużycia pamięci i znacznie szybszym czasem wykonania w porównaniu do Pandas.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: Did local Spark behave as expected compared with the single-node engines?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Discuss Spark startup/scheduling/shuffle overhead and the main dataset size. Mention optional larger stress-test sizes only if you used them.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 7: At what dataset size or query shape would you move from local processing to a cluster?
FINAL_ANSWER_7 = """
TODO: Write your answer here. State a concrete decision boundary supported by your measurements.
"""
display_answer("Final answer 7", FINAL_ANSWER_7)

In [27]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 8: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_8 = """
Porównanie domyślnego backendu Pandas (NumPy) z backendem PyArrow (`dtype_backend="pyarrow"`) wykazuje znaczące różnice na korzyść PyArrow, szczególnie w zużyciu pamięci.

Backend PyArrow jest znacznie wydajniejszy w przypadku kolumn tekstowych (string) oraz w obsłudze brakujących danych (nulls), ponieważ nie musi rzutować ich na ogólny typ `object` ani używać kosztownego typu `float64` dla kolumn z wartościami NaN. W naszych testach zaobserwowaliśmy, że wariant PyArrow zużywał mniej pamięci szczytowej podczas wczytywania plików Parquet i szybciej wykonywał operacje grupowania (Group-by). Domyślny backend Pandas wymusza większy narzut na konwersję typów podczas czytania z formatu Parquet, który natywnie opiera się właśnie na strukturach Arrow.
"""
display_answer("Final answer 8", FINAL_ANSWER_8)


**Final answer 8**

Porównanie domyślnego backendu Pandas (NumPy) z backendem PyArrow (`dtype_backend="pyarrow"`) wykazuje znaczące różnice na korzyść PyArrow, szczególnie w zużyciu pamięci.

Backend PyArrow jest znacznie wydajniejszy w przypadku kolumn tekstowych (string) oraz w obsłudze brakujących danych (nulls), ponieważ nie musi rzutować ich na ogólny typ `object` ani używać kosztownego typu `float64` dla kolumn z wartościami NaN. W naszych testach zaobserwowaliśmy, że wariant PyArrow zużywał mniej pamięci szczytowej podczas wczytywania plików Parquet i szybciej wykonywał operacje grupowania (Group-by). Domyślny backend Pandas wymusza większy narzut na konwersję typów podczas czytania z formatu Parquet, który natywnie opiera się właśnie na strukturach Arrow.